Привет, Даниил. Да, удобно на ты :)

Буду благодарен любой обратной связи даже по мелким недочетам: нейминг, какой-то странное использование написание методов и тд. Перехожу на Python c Dart, было бы круто узнать что-то новое

# Исправления:
- Нашел ошибки в реализации модели LSTM. Теперь она предугадывает значение следущего токена, что пофиксило ошибку пустого вывода;
- Добавил замер метриги rouge для LSTM модели;
- Добавил валидацию Transformer;
- Изменил стратегию выбора следующего токена для LSTM на top-k стратегию;
# Вопросы:
1) Корректно ли заменять, например, ссылки на такую маску - [LINK], чтобы модель понимала контекст, что в данном месте есть ссылка? Или это приведет к ошибкам в выводе?

# Фиксы в коде вне ipynb:
1) utils.py
```python
torch.device("gpu") -> return torch.device("cuda")
```
Остальные изменения: https://github.com/slermo/1_ynd_lstm_vs_transformer/pull/2

In [10]:
%load_ext autoreload
%autoreload 2
from src.data_utils import DataUtils
import yaml
import pandas as pd
from src.utils import my_device
from src.next_token_dataset import NextTokenDataset
from transformers import BertTokenizerFast
from torch.utils.data import DataLoader
import torch
from src.lstm_model import LstmModel
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM
from src.transformer import transformer_make_prediction
import evaluate
from src.eval_transformer_pipeline import eval_transformer_pipeline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
# Загрузка конфига
with open("configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

In [ ]:
# Создание csv файлов, если есть только исходники
# DataUtils.samples_create(config['dataset'])

1280398 160050 160050


In [6]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

df_train = pd.read_csv(config['dataset']['path'] + '/train.csv')
df_val = pd.read_csv(config['dataset']['path'] + '/val.csv')

train_dataset = NextTokenDataset(df_train["text"].tolist(), tokenizer, max_len=16)
val_dataset = NextTokenDataset(df_val["text"].tolist(), tokenizer, max_len=16)

# num_workers = 0 - чтобы не было дедлоков
# The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=0)


In [8]:
for batch in train_loader:
    print(batch["input_ids"].shape)
    break

torch.Size([256, 15])


In [ ]:
# Превал обучение модели из-за параметров системы, но уже можно отследить динамику и примеры по 8 эпохам
from src.eval_lstm_pipeline import eval_lstm_pipeline
trns_model = LstmModel(vocab_size=tokenizer.vocab_size, hidden_dim=128).to(my_device())

eval_lstm_pipeline(
    config=config,
    model=trns_model,
    tokenizer=tokenizer,
    train_loader=train_loader,
    val_loader=val_loader
)

|  1/10 | Train: 6.6344 | Val: 6.4194 | Acc: 0.0921 | R1: 0.0270 | R2: 0.0000 | RL: 0.0270 |
Generated example: the meaning of life just it on so im out
----------------------------------------------------------------------------------------------------
|  2/10 | Train: 6.3482 | Val: 6.2965 | Acc: 0.0999 | R1: 0.0360 | R2: 0.0000 | RL: 0.0360 |
Generated example: the meaning of life a time go ande a days that
----------------------------------------------------------------------------------------------------
|  3/10 | Train: 6.2656 | Val: 6.2442 | Acc: 0.1030 | R1: 0.0397 | R2: 0.0000 | RL: 0.0397 |
Generated example: the meaning of lifett for at today have my
----------------------------------------------------------------------------------------------------


|  4/10 | Train: 6.2241 | Val: 6.2157 | Acc: 0.1047 | R1: 0.0424 | R2: 0.0000 | RL: 0.0424 |
Generated example: the meaning of life thes no to all to in right
----------------------------------------------------------------------------------------------------


|  5/10 | Train: 6.2002 | Val: 6.1995 | Acc: 0.1054 | R1: 0.0439 | R2: 0.0000 | RL: 0.0439 |
Generated example: the meaning of life me for and didn play the that
----------------------------------------------------------------------------------------------------


|  6/10 | Train: 6.1845 | Val: 6.1849 | Acc: 0.1065 | R1: 0.0441 | R2: 0.0000 | RL: 0.0441 |
Generated example: the meaning of life now was sad we be to my
----------------------------------------------------------------------------------------------------


|  7/10 | Train: 6.1722 | Val: 6.1727 | Acc: 0.1070 | R1: 0.0446 | R2: 0.0000 | RL: 0.0446 |
Generated example: the meaning of life that are the with of httpww
----------------------------------------------------------------------------------------------------


|  8/10 | Train: 6.1620 | Val: 6.1711 | Acc: 0.1071 | R1: 0.0458 | R2: 0.0000 | RL: 0.0458 |
Generated example: the meaning of life its on so outside is i it be
----------------------------------------------------------------------------------------------------


KeyboardInterrupt: 

In [13]:
trns_model_name = "distilgpt2" # лёгкая версия GPT-2
tokenizer = AutoTokenizer.from_pretrained(trns_model_name)
trns_model = AutoModelForCausalLM.from_pretrained(trns_model_name).to(my_device())

In [ ]:

# Метрики transformer без предобучения
eval_transformer_pipeline(
    trns_model, 
    val_loader,
    tokenizer)

distilgpt2 Metrics on val dataset
|AvgLoss: 11.7514 | Acc: 0.0453 | R1: 0.0020 | R2: 0.0004 | RL: 0.0020 |


(11.751393529934624,
 0.04531708840987191,
 {'rouge1': np.float64(0.002028993915591849),
  'rouge2': np.float64(0.0004219514734978652),
  'rougeL': np.float64(0.0020258029484833536),
  'rougeLsum': np.float64(0.0020233781110069703)})

Значения метрик на последней эпохе LSTM модели:

|  8/10 | Train: 6.1620 | Val: 6.1711 | Acc: 0.1071 | R1: 0.0458 | R2: 0.0000 | RL: 0.0458 |

Значения метрик предобученной transformer модлели `distilgpt2`

|AvgLoss: 11.7514 | Acc: 0.0453 | R1: 0.0020 | R2: 0.0004 | RL: 0.0020 |

Хоть и LSTM модель показала лучшие метрики, но она была обучена на этом корпусе текста. Transformer не дообучалась и формально по метрикам слабее, но тексты выглядят более грамотными. Разница в ROUGE подтверждает, что без дообучения предобученная модель не способна эффективно адаптироваться под новую задачу.

In [16]:
# Тестирование моделей
df_test = pd.read_csv(config['dataset']['path'] + '/test.csv')
test_dataset = NextTokenDataset(df_test["text"].tolist(), tokenizer, max_len=16)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=0)

In [14]:

print('Значения метрик тестового датасета: ')
eval_transformer_pipeline(
    trns_model, 
    test_loader,
    tokenizer)

Значения метрик тестового датасета: 


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


distilgpt2 Metrics on val dataset
|AvgLoss: 11.7511 | Acc: 0.0447 | R1: 0.0019 | R2: 0.0005 | RL: 0.0019 |


In [15]:

print('Примеры генерации')
for i in range(5):
    text = df_test.iloc[i]["text"]
    words = text.split()
    split_point = len(words) * 3 // 4
    prompt = words[:split_point]
    prompt_str = " ".join(prompt)
    reference = " ".join(words[split_point:]) # Часть для сравнения с генерацией

    print('PROMPT: ' + prompt_str)

    inputs = tokenizer(
        prompt_str, 
        return_tensors='pt',
        padding=True,
        truncation=True
    ).to(my_device())
    
    # Генерация
    with torch.no_grad():
        generated_ids = trns_model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=20,
            do_sample=True,
            top_k=50,
            top_p=0.9,
            temperature=0.8,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    print('TNSF result: '+ generated_text.replace("\n", " ").strip())
    print('-'*80)

Примеры генерации
PROMPT: joejonas1fan1 glad to hear it im alright just cant sleep lol really tired but my nerves wont
TNSF result: joejonas1fan1 glad to hear it im alright just cant sleep lol really tired but my nerves wont go down lol im going to sleep lol im going to sleep lol im going to sleep lol im going
--------------------------------------------------------------------------------
PROMPT: joeymcintyre cant wait to see you in denver this time im standing by you please make sure
TNSF result: joeymcintyre cant wait to see you in denver this time im standing by you please make sure you dont see me here in denver this time im standing by you please make sure you dont see
--------------------------------------------------------------------------------
PROMPT: first load of
TNSF result: first load of new scripts from the web. This is the first time that we've ever created a new script.
--------------------------------------------------------------------------------
PROMPT: boooo its 